# 02. Agent와 Runner

**Runner**는 에이전트 루프를 돌리는 실행기입니다.

- **동기**: `Runner.run_sync(agent, input)` — 스크립트/테스트에 적합
- **비동기**: `await Runner.run(agent, input)` — Jupyter/비동기 앱에 적합

**에이전트 루프** 동작:
1. LLM 호출 (에이전트 설정 + 메시지 히스토리)
2. 응답에 툴 호출이 있으면 실행 후 결과를 메시지에 추가하고 1번으로
3. 핸드오프가 있으면 대상 에이전트로 바꾸고 1번으로
4. **final_output**이 나오면 루프 종료 후 반환
5. `max_turns`로 최대 턴 수 제한 가능


### Runner 실행 방식 3종 비교

| 메서드 | 호출 방식 | 특징 | 언제 쓰나 | 핵심 차이 |
|---|---|---|---|---|
| `Runner.run(...)` | `await Runner.run(...)` | 비동기적, 에이전트 루프 자동 실행, 도구 & 핸드오프 지원 | FastAPI, Jupyter, 서버 환경 | `await` 필요, 비동기 환경용 |
| `Runner.run_sync(...)` | `Runner.run_sync(...)` | 동기 호출, 내부적으로 이벤트 루프를 만들어 `Runner.run()` 실행 | 일반 Python 스크립트, 테스트 | `await` 불필요, 동기 환경용 |
| `Runner.run_streamed(...)` | `Runner.run_streamed(...)` + `async for` | 토큰 단위 실시간 출력, 중간 이벤트를 실시간 수신 | 실시간 채팅 UI, 챗봇 서비스 | 답변을 조각조각 실시간 수신 |

- Jupyter notebook은 기본적으로 이벤트 루프가 이미 실행 중이므로 `await Runner.run(...)` 사용
- `Runner.run_streamed(...)` 는 `await` 없이 호출하며, 반환된 결과의 `stream_events()`를 `async for`로 순회하며 소비


## 1. 동기 실행 (Runner.run_sync)

`Runner.run_sync()`는 **동기 호출** 방식으로, 내부적으로 새 이벤트 루프를 만들어(`asyncio.run()` 방식) `Runner.run()`을 실행합니다.
`await`가 필요 없으므로 **일반 Python 스크립트, 테스트** 환경에 적합합니다.

단, Jupyter처럼 **이미 이벤트 루프가 실행 중인 환경**에서는 새 루프를 만들 수 없어 예외가 발생합니다.
아래 셀의 try/except는 이를 확인하기 위한 것입니다. (Jupyter에서는 `await Runner.run()`을 사용해야 함)

## 2. 비동기 실행 (Runner.run)

`await Runner.run(...)`은 **비동기 호출** 방식으로, 에이전트 루프를 자동 실행하며 **도구(tool)와 핸드오프(handoff)** 를 지원합니다.
**Jupyter, FastAPI** 등 이미 이벤트 루프가 돌고 있는 환경에 적합합니다.

일반 스크립트에서 사용하려면 `asyncio.run(main())`으로 감싸서 실행합니다.

```python
import asyncio
from agents import Agent, Runner

async def main():
    agent = Agent(name="Assistant", instructions="Reply in one short sentence.")
    result = await Runner.run(agent, "What is the capital of France?")
    print(result.final_output)

asyncio.run(main())
```

아래 예제는 도구를 등록한 에이전트를 `await Runner.run()`으로 실행합니다.

In [ ]:
def NumberTool(n: int) -> int:

## 3. 스트리밍 실행 (Runner.run_streamed)

`Runner.run_streamed()`는 LLM의 응답을 **토큰 단위로 실시간 수신**할 수 있어,
실시간 채팅 UI처럼 **답변이 생성되는 과정을 그대로 보여주는 서비스**에 적합합니다.

사용 방법:
1. `result = Runner.run_streamed(agent, input)` — `await` 없이 호출 (즉시 `RunResultStreaming` 반환)
2. `async for event in result.stream_events():` — 이벤트를 실시간으로 순회
3. 스트리밍이 끝나면 `result.final_output`으로 최종 결과 접근 가능

### 스트림 이벤트 종류

| 이벤트 타입 | 내용 | 용도 |
|---|---|---|
| `raw_response_event` | LLM이 생성하는 **토큰 단위 델타** | 타자기 효과의 실시간 텍스트 출력 |
| `run_item_stream_event` | 메시지 완성, 도구 호출/결과 등 **의미 단위 이벤트** | 진행 상황 표시 ("도구 실행 중...") |
| `agent_updated_stream_event` | 핸드오프로 **에이전트가 교체**될 때 | 현재 응답 주체 표시 |

### 3-1. 토큰 단위 실시간 출력

`raw_response_event` 중 텍스트 델타(`ResponseTextDeltaEvent`)만 골라 출력하면,
챗봇 UI처럼 글자가 실시간으로 찍히는 효과를 얻을 수 있습니다.

In [ ]:
# await 없이 호출 → RunResultStreaming 객체 반환
# 이벤트 스트림을 async for로 순회하며 토큰 단위로 출력

In [ ]:
# 스트리밍 종료 후에는 final_output으로 전체 결과에 접근 가능

### 3-2. 의미 단위 이벤트 (run item events)

토큰 델타 대신 **"도구가 호출됨", "메시지가 완성됨"** 같은 의미 단위의 진행 상황을 받아
사용자에게 상태를 알려줄 수도 있습니다.

In [ ]:
def pick_joke_count() -> int:

## 4. max_turns

루프가 무한히 돌지 않도록 최대 턴 수를 제한합니다.
턴 수를 초과하면 기본적으로 `MaxTurnsExceeded` 예외가 발생합니다.
(예외 대신 메시지로 처리하려면 `03_error_handling.py`의 error_handlers 사용)

In [ ]:
# max_turns 초과 시 예외 발생 → 프로그램 중단

### 실습 문제

1. **스트리밍 시인 에이전트**:
   사용자가 요청한 주제로 5줄 내외의 짧은 시를 짓는 `Poet` 에이전트를 만들고,
   `Runner.run_streamed()`로 실행하여 **토큰 단위로 실시간 출력**(타자기 효과)하세요.
   (힌트: `raw_response_event` + `ResponseTextDeltaEvent`)

2. **진행 상황 표시**:
   주사위를 굴려 1~6 사이의 숫자를 반환하는 도구 `roll_dice()`를 등록한 에이전트를 만들고,
   `run_item_stream_event`를 이용해 다음과 같은 **의미 단위 진행 상황**을 출력하세요.
   - 도구가 호출되면: `"-- 도구 호출됨"`
   - 도구 결과가 나오면: `"-- 도구 결과: <값>"`
   - 메시지가 완성되면: 완성된 메시지 출력

3. **max_turns 제한**:
   2번에서 만든 에이전트에게 "주사위를 5번 굴려주세요"라고 요청하되 `max_turns=2`로 제한하고,
   `MaxTurnsExceeded` 예외를 잡아 `"턴 수 초과! max_turns를 늘려주세요."`를 출력하세요.


### 테스트 입력 예시

* `"바다에 대한 시를 써주세요."`
  👉 스트리밍으로 시가 한 글자씩 출력

* `"주사위를 굴려서 나온 숫자로 짧은 농담을 만들어줘."`
  👉 도구 호출됨 → 도구 결과 → 메시지 완성 순서로 출력

* `"주사위를 5번 굴려주세요."` (max_turns=2)
  👉 `"턴 수 초과! max_turns를 늘려주세요."` 출력